# 第15章 自分のコードに本物のGPT-2を入れる

OpenAIが公開した1億2400万個の数を、自作GPTに流し込む

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/15-gpt2.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：GPT-2と同じ形の自作GPTを書き、Hugging FaceからダウンロードしたGPT-2の重みを読み込んで、本物と**同じ出力**が出ることを確かめる
- **身につく数学・理論**：GPT-2の構造（Q・K・Vをまとめた計算、重み共有、GELUの近似） ／ マルチヘッドを1回の行列計算でまとめてやる方法 ／ 事前学習とは何か（データ・計算量） ／ スケーリング則とChinchillaの法則 ／ 計算量 $\approx 6ND$ と、学習に必要なメモリの見積もり
- **新しく使うPython・PyTorch**：`nn.ModuleDict` ／ `state_dict` の中身を調べる・コピーする ／ `.split` と `.view` と `.transpose` で形を組み替える ／ `torch.allclose` で一致を確かめる

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（CPUでOK） |
| 所要時間目安 | 90〜120分 |
| 難易度 | ★★★★☆ |

> **✅ CPUでそのまま実行できます**：上のセルから順に再生ボタン（▶）または `Shift + Enter` で実行してください。

> **🎯 この章でつくるもの**
>
> この章は、あなたの自作GPTの**答え合わせ**です。2019年にOpenAIが公開したGPT-2（一番小さい版、約1億2400万パラメータ）と同じ形のGPTを、第12章のコードを少し書き換えて作ります。そこにGPT-2の公開された重みを読み込むと……あなたのコードが、本物のGPT-2と**小数点以下まで同じ出力**を出します。「自分のコードは本物と同じものだ」と、数字で確かめられる瞬間です。

> **🔗 これまでの復習とつながり：第12章・第13章とのつながり**
>
> - **第12章**：トークン埋め込み・位置埋め込み・Pre-LNのTransformerブロック（因果マルチヘッドAttention＋MLP＋残差接続）・出力層を組み合わせて自作GPTを完成させました。
> - **第13章**：モデルの重み一覧は `state_dict()` という「名前 → テンソル」の辞書として取り出したり読み込んだりでき、`sum(p.numel() for p in model.parameters())` で総パラメータ数を数えられることを学びました。
>
> この章では、その自作GPTを本物のGPT-2の設計図にぴったり合わせ、`state_dict()` 経由で1億2400万個の重みを流し込んで**数値の一致を証明**します。

**ミナト**：本物の重みを入れるって、なんだかすごいことのように聞こえます。

**ボンゴ先生**：すごいことさ。OpenAIは、何百万円分もの計算をして見つけた「つまみの位置」1億2400万個を、全部公開してくれた。キミがやるのは、その数をキミの設計図の正しい場所に1つずつはめ込むことだ。設計図が1か所でも違えば、ガラクタになる。完璧に合えば、キミのコードがGPT-2になる。パズルの最後のピースだね。

## GPT-2をダウンロードする

> **✨ コツ：実行環境とリソースの目安**
>
> - **環境**：CPUで完走できます（GPU不要）。
> - **ダウンロードとメモリ**：初回に `gpt2` の重み（約550MB）をHugging Faceから取得します。必要RAMは約2〜3GBです。
> - **所要時間**：読み込みとロジット比較は数十秒、40トークン生成はCPUでも数秒〜十数秒です。
> - **つまずいたら**：ダウンロードに失敗するときはネットワーク接続を確認し、時間をおいて再実行してください（[エラー辞典](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)）。推奨ライブラリの版は `requirements-book.txt` を参照してください。

まずは、Hugging Faceの `transformers` で本物のGPT-2を読み込みます。CPUのままで大丈夫です。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, AutoTokenizer

hf = GPT2LMHeadModel.from_pretrained("gpt2")
tok = AutoTokenizer.from_pretrained("gpt2")
hf.eval()
cfg = hf.config
print("層の数：", cfg.n_layer, " ヘッド：", cfg.n_head, " 幅：", cfg.n_embd)
print("語彙：", cfg.vocab_size, " 窓の大きさ：", cfg.n_positions)
print(f"パラメータの数：{sum(p.numel() for p in hf.parameters()):,}")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
層の数： 12  ヘッド： 12  幅： 768
語彙： 50257  窓の大きさ： 1024
パラメータの数：124,439,808
```

</details>

12層・12ヘッド・幅768。あなたの第13章のGPT（4層・4ヘッド・幅128）を、縦にも横にも大きくしたものです。構造は同じはずです。

#### 📝 確認問題（`ch15-q01` · Lv.1 · GPT-2 small のハイパーパラメータと1ヘッドあたりの次元数）

GPT-2 small の設定は「層数 `n_layer=12`、ヘッド数 `n_head=12`、埋め込み幅 `n_embd=768`」です。マルチヘッドAttentionの **1ヘッドあたりの次元数（`hs`）** はいくつですか？

- **a.** `64`（埋め込み幅をヘッド数で割る）
- **b.** `9216`（埋め込み幅とヘッド数を掛け合わせる）
- **c.** `12`（ヘッド数や層数と同じ次元数にする）
- **d.** `768`（各ヘッドが埋め込み幅を丸ごと持つ）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：マルチヘッドAttentionでは、埋め込み幅 `n_embd` を全ヘッドで等分して並列に計算します。
- **ヒント 2**：`hs = n_embd // n_head` に `n_embd = 768` と `n_head = 12` を当てはめて計算しましょう。
- **正解**：`a`

**考え方**：マルチヘッドAttentionは埋め込み次元 `C` を `n_head` 等分して並列に異なる視点から注目し、最後に再び幅 `C` へ結合します。

1. `n_embd = 768`、`n_head = 12`。
1. 1ヘッドの次元数 `hs = 768 // 12 = 64`。
1. 12ヘッド分の出力 `[B, 12, T, 64]` を横につなげ直すと `[B, T, 768]` に戻る。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/15-gpt2.html#ex-ch15-q01

</details>

## 重みの名前と形を見る

モデルの中身（`state_dict`）を見てみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
sd_hf = hf.state_dict()
for k in list(sd_hf.keys())[:16]:
    print(f"{k:42s} {tuple(sd_hf[k].shape)}")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
transformer.wte.weight                     (50257, 768)
transformer.wpe.weight                     (1024, 768)
transformer.h.0.ln_1.weight                (768,)
transformer.h.0.ln_1.bias                  (768,)
transformer.h.0.attn.c_attn.weight         (768, 2304)
transformer.h.0.attn.c_attn.bias           (2304,)
transformer.h.0.attn.c_proj.weight         (768, 768)
transformer.h.0.attn.c_proj.bias           (768,)
transformer.h.0.ln_2.weight                (768,)
transformer.h.0.ln_2.bias                  (768,)
transformer.h.0.mlp.c_fc.weight            (768, 3072)
transformer.h.0.mlp.c_fc.bias              (3072,)
transformer.h.0.mlp.c_proj.weight          (3072, 768)
transformer.h.0.mlp.c_proj.bias            (768,)
transformer.h.1.ln_1.weight                (768,)
transformer.h.1.ln_1.bias                  (768,)
```

</details>

第12章の部品と見比べると、対応がわかります。

| GPT-2の名前 | 第12章の部品 | 意味 |
|---|---|---|
| `wte` | `tok_emb` | 文字（トークン）の埋め込み（word token embedding） |
| `wpe` | `pos_emb` | 位置の埋め込み（word position embedding） |
| `h.0`, `h.1`, … | `blocks[0]`, … | Transformerブロック |
| `ln_1`, `ln_2` | `ln1`, `ln2` | LayerNorm |
| `attn.c_attn` | 各ヘッドの `key`, `query`, `value` | **Q・K・Vをまとめた**全結合層（768 → 2304 = 768×3） |
| `attn.c_proj` | `proj` | ヘッドの結果を混ぜる層 |
| `mlp.c_fc`, `mlp.c_proj` | `ffwd.net[0]`, `ffwd.net[2]` | フィードフォワード（768 → 3072 → 768） |

違いは主に3つです。

1. **Q・K・Vを1つの全結合層でまとめて作る**（`c_attn` の出力が2304 = 768×3）。そして、ヘッドを別々のクラスにせず、1回の計算で全ヘッドをまとめて処理する
2. **Q・K・Vにもバイアスがある**（第12章は `bias=False` でした）
3. **出力層の重みを、トークンの埋め込み `wte` と共有している**（weight tying）。`state_dict` には `lm_head.weight` という名前もありますが、中身は `wte` と同じ表です（パラメータの数を数えるときは、1つとして数えられます）

さらに、GELUは計算の速い「tanhを使った近似版」が使われています。これらを合わせれば、読み込めるはずです。

#### 📝 確認問題（`ch15-q02` · Lv.2 · Q・K・Vをまとめた c_attn の出力次元数）

GPT-2では Query・Key・Value を1つの全結合層 `c_attn` でまとめて作ります。埋め込み幅 `n_embd = 768` のGPT-2 smallでは、`c_attn` の出力次元数はいくつになりますか？（整数で答えてください）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`c_attn` は、Query・Key・Value の3種類のベクトルを1つの全結合層でまとめて出力します。
- **ヒント 2**：Q・K・V それぞれが幅 `n_embd = 768` なので、3本分の合計次元 $3 \times 768$ を計算しましょう。
- **正解**：`2304`

**考え方**：Q・K・Vそれぞれ幅 `n_embd` のベクトルが必要なため、`c_attn` は `n_embd → 3 * n_embd` の線形変換になります。

1. `n_embd = 768`。
1. Q・K・Vの3つ分なので $3 \times 768 = 2{,}304$。
1. 計算後に `.split(768, dim=2)` で768ずつ3つに切り分ける。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/15-gpt2.html#ex-ch15-q02

</details>

## GPT-2と同じ形のGPTを書く

第12章のGPTを、上の3つの違いに合わせて書き直します。マルチヘッドAttentionは、ヘッドごとにクラスを作る代わりに、**形の組み替え**で全ヘッドを一度に計算します。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.n_head = n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)     # Q・K・Vをまとめて作る
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)), persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        hs = C // self.n_head                           # 1ヘッドの大きさ
        q, k, v = self.c_attn(x).split(C, dim=2)        # 最後の軸をC個ずつに切る（2304→768・768・768）。文字列のsplitとは別物
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)   # [B, ヘッド, T, hs]
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)
        att = q @ k.transpose(-2, -1) / math.sqrt(hs)   # 全ヘッドの点数を一度に
        att = att.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = att @ v                                     # [B, ヘッド, T, hs]
        y = y.transpose(1, 2).contiguous().view(B, T, C)    # ヘッドを横につなげ直す
        return self.c_proj(y)


class MLP(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU(approximate="tanh")         # GPT-2のGELU（近似版）
        self.c_proj = nn.Linear(4 * n_embd, n_embd)

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT2(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd, n_head, n_layer):
        super().__init__()
        self.block_size = block_size
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(vocab_size, n_embd),
            wpe=nn.Embedding(block_size, n_embd),
            h=nn.ModuleList([Block(n_embd, n_head, block_size) for _ in range(n_layer)]),
            ln_f=nn.LayerNorm(n_embd),
        ))
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.transformer.wte.weight    # 重み共有（weight tying）

    def forward(self, idx):
        B, T = idx.shape
        x = self.transformer.wte(idx) + self.transformer.wpe(torch.arange(T, device=idx.device))
        for block in self.transformer.h:
            x = block(x)
        return self.lm_head(self.transformer.ln_f(x))

> **💡 理論のポイント：ヘッドをまとめて計算する形の組み替え**
>
> 第12章では、4つのヘッドを、`Head` クラスから作った4つの部品で別々に計算していました。わかりやすいけれど、GPUにとっては「小さな仕事を何回も」頼まれるより「大きな仕事を1回でドカンと」頼まれるほうが、GPUコアを無駄なく使い切れてずっと高速なのです。
>
> Q・K・Vを作る3つの全結合層を `c_attn = nn.Linear(n_embd, 3 * n_embd)` の1つに合体させているのも同じ理由です。GPUに3回命令を出すオーバーヘッドを削り、1回の大きな行列積で計算してから、あとで3つに切り分けます。
>
> さらに、Q（形 `[B, T, 768]`）を `view` で `[B, T, 12, 64]`（12ヘッド×64）に切り分け、`transpose(1, 2)` で `[B, 12, T, 64]` に並べ替えます。こうすると、テンソルの `@` が「バッチごと・ヘッドごと」に行列の掛け算をしてくれるので、12ヘッド分の計算が**1回**で済みます。最後に逆の操作で `[B, T, 768]` に戻します（`contiguous()` は、並べ替えたテンソルをメモリ上できれいに並べ直すおまじない）。
>
> 計算の中身は、第11〜12章とまったく同じです。

> **🐍 Python・PyTorchのポイント：nn.ModuleDict と persistent=False**
>
> - `nn.ModuleDict(dict(wte=…, wpe=…))` は、部品に**名前をつけて**入れておく箱です。`dict(wte=1)` は `{"wte": 1}` と同じ意味の、辞書の別の書き方です。こうしておくと、部品の名前が `transformer.wte.weight` のようになり、GPT-2の重みの名前（上で見た一覧）と**一字一句同じ**になります。名前が同じなら、次の節で「同じ名前どうしをコピー」するだけで済みます
> - `register_buffer(…, persistent=False)` の `persistent=False` は、「`tril`（カンニング防止の三角形）は、保存する中身（`state_dict`）に入れない」という指定です。GPT-2の重みの一覧には `tril` がないので、これを付け忘れると、コピーのときに `KeyError: 'transformer.h.0.attn.tril'` になります

> **💡 理論のポイント：重み共有（weight tying）：同じ辞書の表と裏**
>
> 入力側の埋め込み `wte`（トークンID $\to$ 意味ベクトル）と、出力層 `lm_head`（意味ベクトル $\to$ 各トークンの点数）は、どちらも「語彙数 $\times$ 幅」の大きな行列です。
>
> これは英和辞典と和英辞典のようなものです。「単語から意味を引く」作業と、「意味から単語を引く」作業は表裏一体ですよね。GPT-2はこの2つを**まったく同じ表**にしています。「『猫』のベクトルに近いベクトルが計算されたら、次は『猫』の点数を高くする」という、入口と出口の対応関係を自然に表せるうえ、パラメータが3860万個（50257 $\times$ 768）も丸ごと節約できます。

<details><summary><b>基礎の補足：理解のための最小コード：重み共有（`wte.weight` == `lm_head.weight`）の仕組み</b>（クリックで開く）</summary>

`self.lm_head.weight = self.transformer.wte.weight` と1行書くと、コピーではなく**メモリ上のまったく同じテンソル**を2つの名前から参照します。

```python no-run
# 入力の埋め込みと出力層が「同じオブジェクト」を指している
assert mine.lm_head.weight is mine.transformer.wte.weight   # True！

# どちらも形は (50257, 768) → 約 3,860 万個
# parameters() は同一オブジェクトを1回しか数えないため、3,860 万パラメータを丸ごと節約！
```

入口（トークンID → 768次元ベクトル）と出口（768次元ベクトル → 50,257語のロジット）で同じ表を共有することで、パラメータ数を大きく抑えながら意味空間を統一しています。

</details>

#### 📝 確認問題（`ch15-q03` · Lv.3 · マルチヘッド一括計算のための view・transpose・contiguous）

自作GPT-2の `CausalSelfAttention` では、形 `[B, T, 768]` の `q` を `q.view(B, T, 12, 64).transpose(1, 2)` として形 **`[B, 12, T, 64]`** に並べ替えます。`transpose(1, 2)` で「ヘッド軸（12）」を「時間軸（T）」より前に持ってくる理由は何ですか？

- **a.** 入力文のトークン長 `T` をヘッド次元の64と入れ替えることで、あらゆる文章の長さを64トークンに固定するため
- **b.** PyTorchの仕様上、テンソルの軸1（2番目の軸）に12より大きな値を置くと行列積で実行時エラーになるから
- **c.** 後ろ2軸を `[T, 64]` にそろえ、全バッチ・全12ヘッドのスコア計算を1回の行列積でまとめて実行するため
- **d.** 12個のヘッドの特徴ベクトルを時間軸に沿って足し合わせ、計算量を減らすために1ヘッドへ統合してしまうため

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：PyTorchのバッチ行列積演算子 `@` は、テンソルのどの軸どうしを掛け算するルールだったか思い出しましょう。
- **ヒント 2**：`@` は一番後ろの2軸を行列積し、それより前の軸を並列バッチとして扱います。`[B, 12, T, 64] @ [B, 12, 64, T]` の結果の形を確認しましょう。
- **正解**：`c`

**考え方**：後ろ2軸を `[T, hs]` にそろえておけば、前の軸 `[B, n_head]` はすべて独立した並列バッチとして扱われ、12ヘッドのAttentionがたった1回の `@` で完了します。

1. `q`: `[B, T, 768]` → `.view(B, T, 12, 64)` でヘッドを分離。
1. `.transpose(1, 2)` → `[B, 12, T, 64]`（後ろ2軸が `[T, 64]`）。
1. `k.transpose(-2, -1)` → `[B, 12, 64, T]`。
1. `q @ k.transpose(-2, -1)` → `[B, 12, T, T]`（全12ヘッド分のスコア表を一括計算！）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/15-gpt2.html#ex-ch15-q03

</details>

#### 📝 確認問題（`ch15-q04` · Lv.3 · 重み共有（Weight Tying）の意味とパラメータ節約量）

自作GPT-2の `self.lm_head.weight = self.transformer.wte.weight`（**重み共有 / weight tying**）によって起きることとして正しいものはどれですか？

- **a.** 全12層のTransformerブロックの重み行列がすべて共有され、モデル全体の層数ぶんメモリが減る
- **b.** 入力埋め込み行列と出力層行列が同じテンソルを共有し、約3,860万個のパラメータを節約できる
- **c.** 重みの数値だけを別メモリへコピーして初期値をそろえるため、学習時のパラメータ総数は変わらない
- **d.** 位置埋め込み `wpe` がトークン埋め込みに統合されるため、語順を表す位置情報が不要になる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`wte.weight` と `lm_head.weight` の行列の形を比べてみましょう。どちらも `(50257, 768)` です。
- **ヒント 2**：Pythonで同じ `Parameter` オブジェクトを代入して共有すると、実体のメモリと総パラメータ数（$50{,}257 \times 768$）はどうなるでしょうか？
- **正解**：`b`

**考え方**：「トークンID → 意味ベクトル」の表（`wte`）と「意味ベクトル → 各トークンのロジット（内積）」の表（`lm_head`）を共有することで、意味の近いトークンが出やすい構造になり、かつ約3,860万パラメータを丸ごと節約できます。

1. `wte.weight` の形：`(50257, 768)`（約3,860万個）。
1. `lm_head` は `nn.Linear(768, 50257, bias=False)` なので `lm_head.weight` の形も `(50257, 768)`。
1. 同じパラメータオブジェクトを共有すると、`model.parameters()` は重複を除いて返すため総数が1.24億に収まる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/15-gpt2.html#ex-ch15-q04

</details>

*(この節の追加問題 `ch15-q05` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/15-gpt2.html で自動採点・復習できます)*

## 重みを流し込む

同じ設定で自作GPT-2を作り、GPT-2の重みを1つずつコピーします。1つだけ注意点があります。GPT-2の公開コードでは、全結合層の重みを `[入力, 出力]` の向きで持っていますが、PyTorchの `nn.Linear` は `[出力, 入力]` の向きです。だから、該当する重みは **転置**（`.t()`、行と列の入れ替え）してからコピーします。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
mine = GPT2(cfg.vocab_size, cfg.n_positions, cfg.n_embd, cfg.n_head, cfg.n_layer)
mine.eval()
sd = mine.state_dict()
transposed = ("attn.c_attn.weight", "attn.c_proj.weight", "mlp.c_fc.weight", "mlp.c_proj.weight")

copied = 0
with torch.no_grad():
    for k in sd:
        src = sd_hf[k].t() if k.endswith(transposed) else sd_hf[k]
        assert sd[k].shape == src.shape, (k, sd[k].shape, src.shape)
        sd[k].copy_(src)
        copied += 1
print("コピーした重みの数：", copied)

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
コピーした重みの数： 149
```

</details>

`mine.state_dict()` が返す `sd` の中身は、コピーではなく、モデルの**パラメータそのもの**です。だから `sd[k].copy_(src)`（`src` の値で上書き）をすると、`mine` の中身が書き換わります。

`assert` は「この条件が正しくなければ、ここで止まって教えて」という命令です。形が1つでも合わなければエラーになるので、黙って間違ったコピーをすることはありません。

<details><summary><b>基礎の補足：なぜ転置 `.t()` が必要？ HFの `Conv1D` vs PyTorchの `nn.Linear`</b>（クリックで開く）</summary>

Hugging FaceのGPT-2実装は、OpenAIの元コード（TensorFlow由来）に合わせて、全結合層に **`Conv1D`** という独自クラスを使っています。

| 層の種類 | 計算式 | 重み行列の形（例：768 → 2304） | 転置 `.t()` は必要？ |
|---|---|---|---|
| HF GPT-2 の `Conv1D` | $x W + b$ | `[入力, 出力]` ＝ **`(768, 2304)`** | コピー元 |
| PyTorch の `nn.Linear` | $x W^\top + b$ | `[出力, 入力]` ＝ **`(2304, 768)`** | **必要！** |
| バイアス・LayerNorm・埋め込み | そのまま | `(2304,)`, `(768,)`, `(50257, 768)` | **不要** |

特に注意したいのは、正方形の `attn.c_proj.weight`（`(768, 768)`）です。縦横が同じなので、転置し忘れても `assert sd[k].shape == src.shape` を素通りしてしまいます。だからこそ、次の節で「出力ロジットが一致するか」を必ず検証します。

</details>

#### 📝 確認問題（`ch15-q06` · Lv.3 · HF GPT-2（Conv1D）の重み形状と PyTorch nn.Linear の転置）

Hugging Face版GPT-2の `transformer.h.0.attn.c_attn.weight` の形は **`(768, 2304)`**（`[入力, 出力]`）でした。同じ役割を PyTorch の `nn.Linear(768, 2304)` で作ったとき、`mine` 側の `c_attn.weight` の形はどうなりますか？ またコピー時に何が必要ですか？

- **a.** `(768, 2304)`（`[入力, 出力]`）で向きが一致しているため、そのまま変換せずコピーする
- **b.** `(2304,)` の1次元ベクトルになるため、入力軸方向に平均を取って潰してからコピーする
- **c.** `(768, 768)` になるため、HF側の重みから先頭768列だけを切り出してコピーする
- **d.** `(2304, 768)`（`[出力, 入力]`）なので、HF側の重みを `.t()` で転置してコピーする

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：PyTorch標準の `nn.Linear(in_features, out_features)` は、重み行列をどの軸順で保持しているか思い出しましょう。
- **ヒント 2**：`nn.Linear(768, 2304).weight` の形は `[out_features, in_features]` です。HF側の `(768, 2304)` と見比べて必要な操作を考えましょう。
- **正解**：`d`

**考え方**：OpenAIの元祖GPT-2実装（TensorFlow由来）を引き継ぐHFの `Conv1D` 層は `[入力, 出力]`、PyTorch標準の `nn.Linear` は `[出力, 入力]` という向き違いがあるため、4種類の全結合重みだけ `.t()` で転置します。

1. HFの `c_attn.weight`：`[入力=768, 出力=2304]` → `(768, 2304)`。
1. PyTorchの `nn.Linear(768, 2304).weight`：`[出力=2304, 入力=768]` → `(2304, 768)`。
1. `sd_hf[k].t()` で転置して `(2304, 768)` にそろえてから `copy_()` する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/15-gpt2.html#ex-ch15-q06

</details>

#### 📝 確認問題（`ch15-q07` · Lv.2 · k.endswith(transposed) による転置対象の判定）

重みのコピーで使った `k.endswith(transposed)` の動きを確認します。次のコードを実行すると何が表示されますか？

```python
transposed = ("attn.c_attn.weight", "attn.c_proj.weight", "mlp.c_fc.weight", "mlp.c_proj.weight")
keys = [
    "transformer.wte.weight",
    "transformer.h.0.attn.c_attn.weight",
    "transformer.h.0.attn.c_attn.bias",
    "transformer.h.5.mlp.c_proj.weight",
]
flags = [k.endswith(transposed) for k in keys]
print(flags)
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：Pythonの `str.endswith(タプル)` は、文字列がタプル内のどれか1つでも接尾辞として終わっていれば `True` を返します。
- **ヒント 2**：4つのキーそれぞれの末尾を `transposed` の4要素と見比べましょう。埋め込み重みや `.bias` が該当するかに注意してください。
- **期待される出力**：`[False, True, False, True]`

**考え方**：`wte.weight`（埋め込み）と `c_attn.bias`（1次元バイアス）は転置対象外で `False`、全結合層の重み2つは `True` になります。

1. `transformer.wte.weight` → 該当なし → `False`。
1. `transformer.h.0.attn.c_attn.weight` → `attn.c_attn.weight` で終わる → `True`。
1. `transformer.h.0.attn.c_attn.bias` → `.bias` なので `False`。
1. `transformer.h.5.mlp.c_proj.weight` → `mlp.c_proj.weight` で終わる → `True`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/15-gpt2.html#ex-ch15-q07

</details>

## 答え合わせ

同じ文章を入れて、自作GPT-2と本物のGPT-2の出力（ロジット）を比べます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
ids = tok("Hello, I'm a language model,", return_tensors="pt").input_ids
with torch.no_grad():
    logits_mine = mine(ids)
    logits_hf = hf(ids).logits
print("出力の形：", tuple(logits_mine.shape))
print("差は0.001未満？", (logits_mine - logits_hf).abs().max().item() < 1e-3)
print("一致した？", torch.allclose(logits_mine, logits_hf, atol=1e-3))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
出力の形： (1, 8, 50257)
差は0.001未満？ True
一致した？ True
```

</details>

**ボンゴ先生**：一致した！！ 🥁ドコドコドコ！ キミが書いたコードは、本物のGPT-2と、小数点以下まで同じ計算をしている。言い換えれば、**キミはGPT-2を理解した**ということだ。ファインマンの黒板の言葉、「作れないものは理解していない」の逆向きだね。作れたものは、理解している。

（出力の形の8は、この文章が8トークン（`Hello` `,` ` I` `'m` ` a` ` language` ` model` `,`）だったという意味です。50257は語彙の数です。）

<details><summary><b>基礎の補足：`assert` と `torch.allclose` によるロジット検証の意味</b>（クリックで開く）</summary>

コンピュータの小数は有限桁（32ビット浮動小数点数）で計算するため、足し算の順番がわずかに違うだけで末尾の桁に $10^{-6}$ ほどの丸め誤差が出ます。そのため `logits_mine == logits_hf`（完全一致）ではなく、最大絶対誤差が `0.001` 未満か（上の行）と、`torch.allclose(logits_mine, logits_hf, atol=1e-3)`（絶対許容 `atol=1e-3` に加え、既定の相対許容 `rtol=1e-5` を用いて $|a-b| \le \text{atol} + \text{rtol}\,|b|$ を全40万要素で判定）の両方で一致を証明します。

```python no-run
# 実務のテストコードでは assert と組み合わせて自動検証にする
assert torch.allclose(logits_mine, logits_hf, atol=1e-3), "実装または重みコピーにバグあり！"
```

もし1か所でも「転置忘れ」「GELU近似の指定忘れ」「`eval()` の呼び忘れ（Dropout有効）」があると、最大誤差は `0.001` どころか `1.0` 以上に跳ね上がるため、この1行の検証が設計図全体の正しさを強力に保証します。

</details>

#### 📝 確認問題（`ch15-q08` · Lv.2 · torch.allclose による数値検証と eval() の必要性）

自作GPT-2と本物のGPT-2の出力ロジットを比べるとき、`logits_mine == logits_hf` ではなく **`torch.allclose(logits_mine, logits_hf, atol=1e-3)`** を使う理由は何ですか？

- **a.** PyTorchのテンソル型には完全一致を比較する `==` 演算子や `torch.equal` 関数が用意されていないから
- **b.** `torch.allclose` を使うと、一致しなかった要素の値を許容範囲内へ自動的に丸めて一致させてくれるから
- **c.** 浮動小数点演算の順序差などで微小な丸め誤差が出るため、許容誤差 `atol` 以内の実質一致を判定するから
- **d.** 自作GPT-2は高速化のためのランダム近似モデルであり、本物と数値が大きくずれるのが通常の仕様だから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：コンピュータの浮動小数点数では `(0.1 + 0.2) == 0.3` が `False` になることがあります。
- **ヒント 2**：HF版（`Conv1D`）と自作版（`nn.Linear`）で何億回も積和演算を行うと末尾の桁に微小な丸め誤差が生じます。厳密なビット一致ではなく何を確かめるべきでしょうか？
- **正解**：`c`

**考え方**：HF版（`Conv1D`）と自作版（`nn.Linear`）では内部の行列積カーネルや加算順序がわずかに異なるため、`1e-5`〜`1e-4` 程度の浮動小数点丸め誤差が出ます。`allclose` で許容範囲内の一致を確かめます。

1. 同じ数式でも、浮動小数点数の加算順序が違うと末尾の桁に $10^{-6}$ 程度の差が生じる。
1. `torch.equal` は完全ビット一致を要求するため、微小な丸め誤差で `False` になることがある。
1. `torch.allclose(a, b, atol=1e-3)` で全要素の差が 0.001 未満であることを検証できる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/15-gpt2.html#ex-ch15-q08

</details>

## GPT-2に文章を書かせる

自作GPT-2で、文章を生成してみましょう。第3章の自作 `generate` と同じループに、上位k個だけから選ぶ **top-k** を加えたものです。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
@torch.no_grad()
def generate(model, text, n=40, temperature=0.8, top_k=50, seed=0):
    torch.manual_seed(seed)
    idx = tok(text, return_tensors="pt").input_ids
    for _ in range(n):
        logits = model(idx[:, -model.block_size:])[:, -1, :] / temperature
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = -float("inf")    # 上位k個以外は候補から外す
        next_id = torch.multinomial(F.softmax(logits, dim=-1), 1)
        idx = torch.cat([idx, next_id], dim=1)
    return tok.decode(idx[0])

print(generate(mine, "Hello, I'm a language model,"))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
Hello, I'm a language model, and I'm interested in how language is used in the real world. I think the best way to understand it is to look at how people use it in everyday life, and what they mean when they
```

</details>

英語ならすらすら書けます。では、日本語ではどうでしょう？

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
print(generate(mine, "吾輩は猫である。"))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
吾輩は猫である。の�が��������、���、�、�����、��、��、�、�����、
```

</details>

> **💡 理論のポイント：モデルは、読んだものしか書けない**
>
> GPT-2は、主に英語のWebページ（約40GBのテキスト）で学習しました。バイトレベルBPEなので日本語の文字を**表す**ことはできても（第2章）、日本語の文章をほとんど**読んだことがない**ので、まともな続きを書けません。
>
> あなたの漱石GPTが漱石風の文章しか書けないのと、まったく同じ理由です。LLMの能力は、**どんなデータで、どれだけ学習したか**で決まります。モデルの設計図（あなたが書いたコード）は、ほとんど同じなのです。

#### 📝 確認問題（`ch15-q09` · Lv.2 · GPT-2が英語は流暢で日本語は書けない理由）

重みをコピーした自作GPT-2に英語の続きを書かせると非常に自然な英文が出るのに、日本語「吾輩は猫である。」の続きを書かせると文字化けだらけになりました。この根本原因は何ですか？

- **a.** 自作GPT-2の重みコピーに未検出のバグがあり、日本語バイト列を通したときだけ転置が外れてしまうから
- **b.** デコーダ型Transformerという設計図自体が英語の語順に特化しており、原理的に日本語を処理できないから
- **c.** 生成時に指定した `top_k=50` がASCII文字だけを候補に残す設定になっており、日本語トークンが禁止されるから
- **d.** バイトレベルBPEで入出力はできても、事前学習データのほぼ全体が英語で日本語をほとんど学んでいないから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「モデルの構造（アーキテクチャ）」「トークナイザーの仕様」「事前学習データ」の3つを切り分けて考えましょう。
- **ヒント 2**：同じTransformerでも日本語データで学べば日本語が書けます。GPT-2の事前学習コーパス（WebText）がどんな言語で構成されていたか思い出しましょう。
- **正解**：`d`

**考え方**：LLMの言語能力・知識は、ほぼ同じTransformer設計図の上に「どんなデータで、どれだけ事前学習したか」で決まります。

1. GPT-2の学習データ（WebText約40GB）はほぼ英語テキスト。
1. バイトレベルBPEのおかげで日本語の入力自体でクラッシュはしないが、日本語の3バイトの並びを正しく続ける確率分布を学んでいない。
1. 結果として不完全なバイト列が生成され、置換文字だらけになる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/15-gpt2.html#ex-ch15-q09

</details>

## 事前学習：GPT-2はどうやって作られたか

GPT-2の重みは、あなたが第13章でやったのとまったく同じ「次のトークン予測」の学習で作られました。違うのは規模だけです。このように、大量の文章で「次のトークン予測」を学習させることを **事前学習**（pre-training）と呼びます。

| | あなたの漱石GPT | GPT-2（small） | 最近の大型LLM（目安） |
|---|---|---|---|
| パラメータ | 約175万 | 1.24億 | 数百億〜数千億以上 |
| 学習データ | 約100万文字 | 約40GB（数十億トークン） | 数兆〜数十兆トークン |
| 計算機 | Colabの無料GPU 1枚・数分 | 当時の大量のGPU・TPU | 数千〜数万枚のGPUで数か月 |

> **💡 理論のポイント：スケーリング則：大きくすると、予想どおりに賢くなる**
>
> 2020年、OpenAIの研究者たちは、LLMの損失が **モデルの大きさ $N$（パラメータ数）**、**データ量 $D$（トークン数）**、**計算量 $C$** を増やすと、とてもきれいな法則に従って下がることを見つけました。それぞれを10倍にすると、損失が決まった割合ずつ下がっていく（グラフの両軸を対数にすると直線になる、**べき乗則**）。これを **スケーリング則**（scaling law）と言います。
>
> この発見の何がすごいかというと、**小さな実験から、大きなモデルの性能が予想できる**ことです。「このくらいのお金をかければ、このくらい賢くなる」と計算できるので、巨大なモデルへの投資が始まりました。
>
> 2022年には、DeepMindの研究（通称 **Chinchilla**）が、「決まった**学習計算量**で一番賢いモデルを作るには、パラメータ1個あたり**約20トークン**のデータで学習させるのがよい」ことを示しました。それまでのモデルは、大きさのわりにデータが少なすぎた（読書量の割に頭だけ大きかった）のです。
>
> 一方、下の表の `Qwen2.5-0.5B` のように、最近の小さな公開LLMはChinchilla推奨（約98億トークン）の**約1,800倍**ものデータ（18兆トークン）で学習されています。これは**推論効率を優先した過剰学習（overtraining）**です。学習コストは増えても、完成後のモデルが小さいほど何億回もの推論（サービス提供や手元実行）が圧倒的に速く安くなるため、「小さく作って、とことん長く読ませる」設計が主流になっています。

計算量と、学習に必要なメモリを見積もってみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def train_flops(n_params, n_tokens):
    return 6 * n_params * n_tokens          # 学習の計算量 ≈ 6 × N × D

def train_memory_gb(n_params):
    return n_params * 16 / 1e9              # AdamWで学習するときのメモリの目安（1パラメータ16バイト）

models = {"漱石GPT": (1.75e6, 2e7), "GPT-2 small": (124e6, 9e9),
          "Qwen2.5-0.5B": (0.49e9, 18e12), "70Bのモデル": (70e9, 15e12)}
for name, (N, D) in models.items():
    print(f"{name:12s} 計算量 {train_flops(N, D):9.1e} 回  学習メモリ {train_memory_gb(N):9.2f} GB"
          f"  Chinchilla推奨データ {20 * N:8.1e} トークン")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
漱石GPT        計算量   2.1e+14 回  学習メモリ      0.03 GB  Chinchilla推奨データ  3.5e+07 トークン
GPT-2 small  計算量   6.7e+18 回  学習メモリ      1.98 GB  Chinchilla推奨データ  2.5e+09 トークン
Qwen2.5-0.5B 計算量   5.3e+22 回  学習メモリ      7.84 GB  Chinchilla推奨データ  9.8e+09 トークン
70Bのモデル      計算量   6.3e+24 回  学習メモリ   1120.00 GB  Chinchilla推奨データ  1.4e+12 トークン
```

</details>

（データ量は公開情報などからのおおまかな目安です。漱石GPTは、第13章の実験2で学習した約2000万トークン（5000ステップ × 32 × 128）で計算しています。）

> **💡 理論のポイント：6ND と 16バイト**
>
> - **計算量 ≈ $6ND$**：1トークンを学習するごとに、パラメータ1個あたり約6回の掛け算・足し算をします（順方向で約2回、逆伝播で約4回）。T4の実効性能を毎秒10兆回（$10^{13}$）とすると、漱石GPTは数十秒、GPT-2 smallは約1週間、Qwen2.5-0.5Bは約170年（！）。大きなLLMの事前学習が、個人では無理な理由です
> - **メモリ ≈ 16バイト × パラメータ数（＋活性化）**：混合精度でAdamWを使うと、パラメータ1個あたり、重み（2バイト＋4バイトの控え）、勾配（2バイト）、Adamの慣性と歩幅の記録（4バイト×2）で約16バイト必要です。さらに実際の学習では、順伝播の途中結果である**活性化（第16章で詳しく）**のメモリもバッチサイズと文章長に応じて上乗せされます。T4のメモリは約15GBなので、全パラメータを学習できるのは**数億パラメータ**くらいまで。それより大きいモデルを無料GPUで鍛えるには、第19章の **LoRA** という技が必要になります

<details><summary><b>基礎の補足：スケーリング則の数式 $C \approx 6ND$ を1つずつ分解して手計算する</b>（クリックで開く）</summary>

記号の意味を並べると、驚くほどシンプルです。

- **$N$**：モデルのパラメータ数（例：1億 $= 10^8$）
- **$D$**：学習に使う総トークン数（例：20億 $= 2 \times 10^9$）
- **$C$**：学習全体で行う浮動小数点演算の回数（**FLOPs**）

**なぜ係数が「6」になるのか？**
行列の掛け算 $y = x W^\top$ では、1トークンに対して1個の重み $w$ が関わるたびに「掛け算1回＋足し算1回＝**2 FLOPs**」使います。

| フェーズ | 何を計算しているか | 1パラメータ・1トークンあたりの演算量 |
|---|---|---|
| 順伝播（forward） | 次の層の出力 $y = x W^\top$ | **2 FLOPs**（掛け算1＋足し算1） |
| 逆伝播①（backward） | 前の層へ伝える入力の勾配 $\partial L / \partial x$ | **2 FLOPs** |
| 逆伝播②（backward） | 自分の重みを更新する勾配 $\partial L / \partial W$ | **2 FLOPs** |
| **学習1ステップ合計** | **順伝播 ＋ 逆伝播** | **$2 + 4 = 6$ FLOPs** |

**具体的な手計算**：パラメータ数 $N = 10^8$（1億）のモデルを、Chinchillaの目安どおり $D = 20N = 2 \times 10^9$（20億）トークンで学習すると、

$$C \approx 6 \times 10^8 \times (2 \times 10^9) = 12 \times 10^{17} = 1.2 \times 10^{18}\ \text{FLOPs}$$

さらに1パラメータあたりの学習メモリ16バイトの内訳は、**fp16重み（2B）＋fp32控え重み（4B）＋勾配（2B）＋AdamWの1次・2次モーメント（4B＋4B）＝16バイト** なので、1億パラメータなら $10^8 \times 16 / 10^9 = 1.6$ GB になります（実運用ではここにバッチサイズと系列長に応じた活性化メモリが加算されます）。

</details>

> **¥0 無料枠のコツ：事前学習は「借りる」が正解**
>
> 数十億パラメータ以上のモデルを、個人が0円で事前学習するのは不可能です。でも心配いりません。世界中の企業や研究機関が、事前学習済みのモデルを公開しています（Qwen、Llama、Gemma、日本語特化のモデルなど）。**事前学習は借りて、自分の目的に合わせた追加の学習（ファインチューニング）だけを自分でやる**。これが、第5部からのやり方です。

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **`AssertionError: ('transformer.h.0.attn.c_attn.weight', …)`**：形が合っていません。転置が必要な重み（`transposed` の4種類）の名前が合っているか、`n_embd` などの設定が `cfg` と同じか確認しましょう。
> - **`KeyError: 'transformer.h.0.attn.tril'`**：`register_buffer` の `persistent=False` を付け忘れています。
> - **そのほかの `KeyError`（`sd_hf[k]` の行）**：自作モデルの部品の名前が、GPT-2の名前と違っています。`transformer`、`wte`、`h`、`ln_1` などの名前を、上のコードと一字一句同じにしましょう。
> - **一致しない（`False`）**：GELUを `approximate="tanh"` にしたか、`mine.eval()` を呼んだか、確認しましょう。

#### 📝 確認問題（`ch15-q10` · Lv.3 · 学習計算量 C ≈ 6ND の「6」の理由（順伝播2＋逆伝播4））

LLMの事前学習にかかる総計算量（FLOPs）の近似式 **$C \approx 6ND$**（$N$：パラメータ数、$D$：学習トークン数）で、先頭の係数「**6**」はどのような内訳から来ていますか？

- **a.** AdamWオプティマイザが1パラメータの更新ごとに重みとモーメントで合計6バイトのメモリを読み書きするから
- **b.** 1トークン・1パラメータあたり、順伝播で約2 FLOPs、逆伝播で約4 FLOPs（入力勾配2＋重み勾配2）の計6 FLOPs
- **c.** 事前学習では同じコーパスを最低6エポック繰り返して学習させることが経験則として決まっているから
- **d.** 標準的なTransformer言語モデルが最小構成として必ず6層のブロックを積む設計になっているから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：行列の積和演算（掛けて足す）は、1パラメータあたり「掛け算1回＋足し算1回」で何FLOPsになるか考えましょう。
- **ヒント 2**：順伝播の行列積1回に加えて、逆伝播では「前の層へ渡す勾配」と「自分の重みの勾配」の2種類の行列積が必要です。
- **正解**：`b`

**考え方**：線形層の計算では、順伝播で $y = x W^\top$（2 FLOPs/param）、逆伝播で「前の層へ渡す勾配 $\partial L / \partial x$（2 FLOPs）」と「自分の重みの勾配 $\partial L / \partial W$（2 FLOPs）」の2種類の行列積を行うため、合計 $2 + 4 = 6$ FLOPsになります。

1. 順伝播：各パラメータ $w$ につき積和（掛け算1＋足し算1）＝ **2 FLOPs**。
1. 逆伝播①：前の層へ伝える活性化の勾配計算 ＝ **2 FLOPs**。
1. 逆伝播②：重み自身の更新用勾配計算 ＝ **2 FLOPs**。
1. 合計：$2 + 2 + 2 = 6$ FLOPs（推論だけなら順伝播のみなので約 $2ND$）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/15-gpt2.html#ex-ch15-q10

</details>

#### 📝 確認問題（`ch15-q11` · Lv.3 · Chinchillaの法則による推奨学習トークン数（20N）の計算）

Chinchillaの法則では、計算量に対して最も効率よく賢いモデルを作る目安は「**パラメータ1個あたり約20トークン**（$D \approx 20N$）」です。**10億パラメータ（1B）** のモデルに必要な推奨トークン数は何**億**トークンですか？（「億」の前の数だけを整数で答えてください）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：Chinchillaの法則では、推奨トークン数 $D$ がパラメータ数 $N$ に比例する式 $D \approx 20N$ で与えられます。
- **ヒント 2**：パラメータ数 $N = 10\text{億}$ に倍率 $20$ を掛けて、「億」の前に付く数値を求めましょう。
- **正解**：`200` 億トークン

**考え方**：$20 \times 10\text{億} = 200\text{億}$（$2 \times 10^{10}$）トークンになります。

1. パラメータ数 $N = 10$ 億（$10^9$）。
1. Chinchilla目安：$D = 20N = 20 \times 10\text{億} = 200\text{億}$ トークン。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/15-gpt2.html#ex-ch15-q11

</details>

*(この節の追加問題 `ch15-q12`, `ch15-q13` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/15-gpt2.html で自動採点・復習できます)*

**✅ できたらチェック**

- [ ] GPT-2の重みの名前と形を、第12章の部品と対応づけられた
- [ ] Q・K・Vをまとめて計算し、ヘッドを形の組み替えで処理するAttentionを書いた
- [ ] GPT-2の重みを自作GPTに読み込み、出力が一致することを確かめた
- [ ] GPT-2が日本語を書けない理由（学習データ）を説明できる
- [ ] 計算量 $6ND$ と学習メモリ（16バイト×パラメータ）を見積もれた

> **📝 まとめ**
>
> - GPT-2は、第12章のGPTと**同じ構造**。違いはQKVの結合・バイアス・重み共有・GELUの近似だけ
> - ヘッドは `view` と `transpose` でまとめて計算できる（速い）
> - 公開された重みを読み込み、**本物と同じ出力**が出ることで、自作コードの正しさを確かめた
> - **事前学習**＝大規模な「次のトークン予測」。性能はモデル・データ・計算量で予測できる（**スケーリング則**）
> - 学習の計算量 ≈ $6ND$、学習メモリ ≈ 16バイト×パラメータ数。大きな事前学習は「借りる」のが正解

**ボンゴ先生**：これで、キミのGPTは本物と同じものだと証明された。次の章では、このGPTを**速く**する工学を学ぶ。生成を何倍も速くするKVキャッシュ、賢い候補の絞り方、大きなモデルを何枚ものGPUで学習する方法……。研究室や会社で「LLMを動かす係」になったとき、真っ先に役立つ話だ。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/15-gpt2.html
- **次の章**：[第16章 速く・大きく：推論と学習の工学](https://sakunokyumei.github.io/llm-research-textbook/16-scale.html)